In [21]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from scipy.optimize import basinhopping

In [22]:
a = 0.0
start = 2
x_grid = np.linspace(-2, 2, 500)

def potential(x):
    return (x[0]**2 - 1)**2 + a*x[0]

def gradient(x):
    return np.array([
        4*x[0]*(x[0]**2 - 1) + a
    ])



In [23]:
def basin_hopping(f, grad, x0, niter=100, stepsize=1.5):
    path = [np.atleast_1d(x0).astype(float)]

    def record_state(x, fval, accepted):
        path.append(x.copy() if accepted else path[-1].copy())

    result = basinhopping(
        f, x0, niter=niter, stepsize=stepsize, seed=7,
        minimizer_kwargs={"method": "BFGS", "jac": grad},
        callback=record_state
    )
    return result, np.asarray(path)

In [24]:
basin_result, path = basin_hopping(potential, gradient, start)
if not basin_result.success:
    raise RuntimeError(f"Basin hopping failed: {basin_result.message}")
current_history = path[:, 0]
best_x = current_history[0]
best_energy = float(potential([best_x]))
best_history = []

for current_x in current_history:
    current_energy = float(potential([current_x]))
    if current_energy < best_energy:
        best_x = current_x
        best_energy = current_energy
    best_history.append(best_x)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x_grid, [potential([xi]) for xi in x_grid], color="black", linewidth=2)
current_point, = ax.plot([], [], "o", color="red", markersize=12)
best_point, = ax.plot([], [], "o", markeredgecolor="black", markerfacecolor="none", markersize=14)
ax.set(xlabel="x", ylabel="V(x)", title="Basin hopping")


def update(frame):
    current = current_history[frame]
    best = best_history[frame]
    current_point.set_data([current], [potential([current])])
    best_point.set_data([best], [potential([best])])
    return current_point, best_point

animation = FuncAnimation(
    fig, update, frames=len(current_history), interval=100, blit=True, repeat=False
)
plt.close(fig)
HTML(animation.to_jshtml())